# GISLR — ASL-LEX rule-based phonology engine, no pose

**What this notebook does.** Diagnostic / research notebook (not a training pipeline stage — nothing here
fits a model). Part of **recognition_2** (TODO §14), following the critique in
`docs/reports/phonology-pipeline-proposal.md`. Answers: **can a rule-based engine, reading only hand and
face landmarks (no MediaPipe pose), identify ASL-LEX 2.0's 18 phonological parameters from a GISLR clip,
well enough to match ASL-LEX's own coding of that clip's gloss?**

Uses `sb.recognize.recognition2.rules` (the engine — new, isolated, never edits `sb.recognize.phonology`
or `aslex`, only imports their pose-independent geometry helpers) and `sb.recognize.recognition2.validate`
(scoring against ASL-LEX ground truth). Restricted to the **233/250 GISLR glosses that map into ASL-LEX**
(the gloss-mapping check earlier in this conversation, 2026-09-26/27) — ASL-LEX is the only ground truth
available, so nothing here can be scored for the other 17.

**Why no pose is a real constraint, not a formality** (user ask, 2026-09-27): the anchor becomes a face
centroid instead of mid-shoulder (weaker under head rotation), and any ASL-LEX category naming a torso/arm
site (`MajorLocation` ∈ {Body, Arm}, most of `MinorLocation`/`SecondMinorLocation`) is structurally
unreachable — `rules.py` reports these as `NOT_COMPUTABLE` rather than guessing, and this notebook keeps
that separate from "reachable but wrong" in the results table.

**Artifacts:**

| path | what |
|---|---|
| `data/cache/gislr_aslex_rules/agreement.csv` | one row per ASL-LEX parameter: coverage + agreement + chance |
| `data/cache/gislr_aslex_rules/records.json` | every scored clip's truth/prediction (for follow-up error analysis) |

**Resumability**: this notebook is fast enough (rule evaluation, no training) to just re-run in full; it
does not need a manifest.


## Setup

In [1]:
# ============================================================
# Imports, tunables, resolved paths
# ============================================================
import json
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sb.core.paths import CACHE_DIR, EXPERIMENTS_DIR, gislr_dir
from sb.core.vocab import load_label_map
from sb.recognize.aslex import gloss_entries, signdata
from sb.recognize.recognition2 import rules, validate

CONFIG_PATH = EXPERIMENTS_DIR / "recognition2" / "configs" / "stage1-rules.json"
THRESHOLDS = rules.load_thresholds(CONFIG_PATH)

MAX_CLIPS_PER_GLOSS = 6  # tunable: raise for a fuller run once the sample-size result looks sane

OUT_DIR = CACHE_DIR / "gislr_aslex_rules"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = gislr_dir()

print(f"data:    {DATA_DIR}")
print(f"config:  {CONFIG_PATH.name}")
print(f"output:  {OUT_DIR}")
print(f"max clips/gloss this run: {MAX_CLIPS_PER_GLOSS}")


data:    C:\Users\user2\.cache\kagglehub\datasets\bracu23101281\gislr-stratified\versions\1
config:  stage1-rules.json
output:  C:\Users\user2\repository\data\cache\gislr_aslex_rules
max clips/gloss this run: 6


## 1. ASL-LEX-mapped glosses and the handshape lookup table

`rules.build_handshape_lookup` reads the **entire** ASL-LEX lexicon (not just the 233 mapped glosses) to
build `{(selected_fingers, flexion, spread, thumb_position, thumb_contact): modal Handshape}` —
Handshape is *defined* by signdataKEY.csv as exactly this combination, so this is a lookup of ASL-LEX's own
coding scheme, not a fit to the glosses this notebook scores against.

In [2]:
# ============================================================
# ASL-LEX-mapped glosses + handshape lookup
# ============================================================
label_map = load_label_map(DATA_DIR)
glosses = sorted(label_map.keys())
ENTRIES = gloss_entries(glosses)
MAPPED = [g for g in glosses if g in ENTRIES]
SIGNDATA = signdata().set_index("EntryID")
LOOKUP, IS_MARKED = rules.build_handshape_lookup()

print(f"GISLR glosses: {len(glosses)}, ASL-LEX-mapped: {len(MAPPED)}")
print(f"handshape lookup: {len(LOOKUP)} sub-feature combinations -> {len(IS_MARKED)} handshapes")


GISLR glosses: 250, ASL-LEX-mapped: 233
handshape lookup: 61 sub-feature combinations -> 35 handshapes


## 2. Score the engine against ASL-LEX ground truth

For each mapped gloss: look up its ASL-LEX ground truth per parameter (`validate.ground_truth_for_gloss`),
run the rule engine on up to `MAX_CLIPS_PER_GLOSS` of its clips, and record both. `rules.extract`/`codes`
never see pose landmarks (they are sliced out of the raw clip before either function is called — see
`rules.py`'s docstring).

In [3]:
# ============================================================
# Run the engine over every mapped gloss's clips
# ============================================================
train = pd.read_csv(DATA_DIR / "train.csv")
rng = np.random.default_rng(0)
RECORDS = []
for g in tqdm(MAPPED, desc="glosses"):
    truth = validate.ground_truth_for_gloss(g, ENTRIES, SIGNDATA)
    clip_rows = train[train["sign"] == g]
    if len(clip_rows) > MAX_CLIPS_PER_GLOSS:
        clip_rows = clip_rows.sample(n=MAX_CLIPS_PER_GLOSS, random_state=rng.integers(1 << 31))
    for _, row in clip_rows.iterrows():
        clip = rules.load_clip(DATA_DIR / row["npz_relpath"])
        feats = rules.extract(clip)
        pred = rules.codes(feats, THRESHOLDS, LOOKUP, IS_MARKED)
        RECORDS.append({"gloss": g, "uid": row["uid"], "truth": truth, "pred": pred})

print(f"scored {len(RECORDS)} clips over {len(MAPPED)} glosses")
(OUT_DIR / "records.json").write_text(json.dumps(RECORDS, indent=1), encoding="utf-8")


glosses:   0%|          | 0/233 [00:00<?, ?it/s]

scored 1398 clips over 233 glosses


1585427

## 3. Per-parameter agreement

Three numbers per parameter, kept separate on purpose (`validate.agreement_table`):

- **coverage**: of the clips with ASL-LEX ground truth, how many the engine could even attempt (not
  `NOT_COMPUTABLE`) — low coverage means "structurally unreachable without pose," not "wrong."
- **agreement**: among attempted clips, how often the engine's code matches ASL-LEX's.
- **chance**: 1 / (number of categories ASL-LEX uses for that parameter, among the 233 mapped glosses).

Expected pattern, per the plan this notebook implements: high agreement on coarse/binary parameters
(SignType, ThumbPosition, RepeatedMovement, Contact, ...), lower on fine-grained ones (Handshape 39-way,
MinorLocation 29-way, SecondMinorLocation 21-way, NonDominantHandshape 31-way).

In [4]:
# ============================================================
# Agreement table
# ============================================================
TABLE = validate.agreement_table(RECORDS)
TABLE = TABLE.sort_values("agreement", ascending=False)
TABLE.to_csv(OUT_DIR / "agreement.csv", index=False)
TABLE


,parameter,n_with_ground_truth,n_attempted,coverage,agreement,chance
10,ulnar_rotation,1314,628,0.477930,0.855096,0.500000
7,spread_change,828,828,1.000000,0.764493,0.500000
9,thumb_contact,1254,1058,0.843700,0.702268,0.500000
5,flexion_change,1248,1248,1.000000,0.661859,0.500000
0,sign_type,1272,1272,1.000000,0.659591,0.200000
8,thumb_position,1320,1111,0.841667,0.625563,0.500000
2,marked_handshape,1272,493,0.387579,0.590264,0.500000
16,contact,1290,1223,0.948062,0.577269,0.500000
12,repeated_movement,1326,1326,1.000000,0.530920,0.500000
11,movement,1038,1038,1.000000,0.493256,0.166667


## 4. Reading the table

- Any parameter with `coverage` well below 1.0 is one this no-pose engine could only sometimes attempt —
  check whether that's the expected pose-dependent gap (`MajorLocation` ∈ {Body, Arm},
  `MinorLocation`/`SecondMinorLocation` outside the face-adjacent set) or something else (e.g. a
  `NOT_COMPUTABLE` from a missing hand in the clip itself).
- `agreement` below `chance` on an attempted parameter is a real miscalibration (a threshold in
  `experiments/recognition2/configs/stage1-rules.json`), not a pose limitation — worth tuning before
  trusting that parameter's rule.
- `Handshape`/`NonDominantHandshape` go through the ASL-LEX-derived lookup table (§1), so their agreement
  is bounded by how well `selected_fingers`/`flexion`/`spread`/`thumb_position`/`thumb_contact` are read
  individually — check those rows first if handshape agreement is low.
